---
title: 'Ray Core · CPU：讲解与代码'
---


# Ray Core：从提交、依赖到有状态并发

**状态：完整教学 Notebook，实验未运行。**

按 **讲解 → 预测 → 代码 → 观察与复述** 阅读，每次完成一个概念单元。
先希望解释：把工作交给其他进程时，谁调度、怎样等结果、状态在哪里、并发发生在哪一层？

[主题入口](README.md) · [下一份：GPU 资源与计算](gpu.ipynb)

在服务器仓库根目录准备 kernel：

```bash
python3.11 -m venv .venv-ray
.venv-ray/bin/python -m pip install -r topics/distributed-runtime/ray/requirements.txt
.venv-ray/bin/python -m ipykernel install --user --name ai-infra-ray --display-name "AI Infra · Ray CPU"
```

VS Code Remote SSH 打开本文件，选择这个远程 kernel；也可以在该环境启动 JupyterLab。
[JupyterLab 4.1 起支持 Markdown 单元中的 Mermaid](https://blog.jupyter.org/posts/2024/jupyterlab-4-1-and-notebook-7-1-are-here/)，下方的流程图在 JupyterLab 与网站中阅读。


## 环境：先把观察的来源记下来

在服务器选择安装了本主题依赖的 Python kernel，按顺序运行单元格。修改代码或参数后先保存 Notebook，再开始新的运行记录。
工作目录应为这个 Notebook 所在的 `topics/distributed-runtime/ray/`；VS Code Remote SSH 已配置 `${fileDirname}`。
JupyterLab 从仓库启动时，先把 kernel 工作目录切换到这个目录。
网站只展示 Notebook 与已保存输出，不启动 kernel。

本轮实验未运行。服务器的双卡 RTX6000 Pro、96 核 CPU 来自你的自述；实际资源以下面的记录为准。
默认 `NUM_CPUS=4` 是教学准入容量，等待时长和任务数也都是教学参数。


In [ ]:
import asyncio
import json
import os
import platform
import socket
import subprocess
import sys
import threading
import time
import traceback
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import ray
from IPython.display import display

TOPIC = Path.cwd().resolve()
assert (TOPIC / "cpu.ipynb").is_file(), "请把 kernel 工作目录切换到 Ray Notebook 所在目录"
REPO = TOPIC.parents[2]
NUM_CPUS = 4
TIMEOUT = 120
owns_runtime = False


### 运行记录也是实验的一部分

每次初始化创建新的 `.runs/` 目录。下面记下实际版本、Git revision、未提交修改和环境；记录不推断机器性能。
`observe()` 给每个运行单元保存成功或异常。失败时关闭本 Notebook 创建的 Ray 运行时，修复后从初始化重新运行。
定义函数和类的单元若有错误，Jupyter 会保留错误输出；保存 Notebook 才能把这些输出取回。


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
RUN_DIR = REPO / ".runs" / "ray-cpu" / stamp
RUN_DIR.mkdir(parents=True, exist_ok=False)
run = {
    "kind": "Ray cpu Notebook learning experiment",
    "notebook": str(TOPIC / "cpu.ipynb"),
    "status": "running",
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "parameters": {"num_cpus": NUM_CPUS, "timeout_seconds": TIMEOUT},
    "environment": {
        "hostname": socket.gethostname(), "platform": platform.platform(),
        "python": platform.python_version(), "python_executable": sys.executable, "ray": ray.__version__,
        "host_cpu_count": os.cpu_count(),
        "cpu_affinity_count": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
        "git_revision": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip(),
        "git_status": subprocess.run(["git", "status", "--porcelain"], cwd=REPO, text=True, capture_output=True, check=True).stdout.splitlines(),
    },
    "clock": "time.perf_counter_ns; event times comparable on this single node",
    "steps": [],
}

def save_record():
    (RUN_DIR / "run.json").write_text(json.dumps(run, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

@contextmanager
def observe(name):
    global owns_runtime
    step = {"name": name, "status": "running", "started_ns": time.perf_counter_ns()}
    run["steps"].append(step)
    try:
        yield step
        step["status"] = "passed"
    except BaseException as error:
        step["status"] = "failed"
        step["error"] = {"type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc()}
        run["status"] = "failed"
        if owns_runtime:
            ray.shutdown()
            owns_runtime = False
        raise
    finally:
        step["finished_ns"] = time.perf_counter_ns()
        save_record()

def event(label):
    return {"label": label, "at_ns": time.perf_counter_ns(), "hostname": socket.gethostname(),
            "pid": os.getpid(), "thread_id": threading.get_ident()}

save_record()
display(run["environment"])
print("Results:", RUN_DIR)


### 建立单节点运行时

这里显式使用 `address="local"` 创建本次实验的运行时，CPU 教学使用 128 MiB 对象存储。
`NUM_CPUS` 是逻辑资源数量，不是物理核绑定或线程上限。
如果 kernel 已连接 Ray，请先完成自己的清理或重启 kernel，再开始本次 Notebook。
资料版本：Ray 2.59.0，查阅 2026-10-02。[官方资源模型](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html)


In [ ]:
with observe("runtime_start") as step:
    if ray.is_initialized():
        raise RuntimeError("kernel 已有 Ray 连接；请先清理自己的运行时或重启 kernel")
    ray.init(address="local", num_cpus=NUM_CPUS, num_gpus=0, include_dashboard=False,
             object_store_memory=128 * 1024**2)
    owns_runtime = True
    run["environment"]["ray_resources"] = ray.cluster_resources()
    step["resources"] = ray.cluster_resources()
    display(step["resources"])


## 1 · Task 与 ObjectRef：先区分提交和等待

普通函数在调用进程执行，返回时拿到值。先运行一个普通 Python 函数，把这个基准放在脑中。

**预测：** 下面的 PID 会对应 Notebook kernel，还是其他进程？


In [ ]:
def normal_square(x):
    return {"value": x * x, "pid": os.getpid()}

normal = normal_square(3)
display({"kernel_pid": os.getpid(), "normal_call": normal})
assert normal["value"] == 9 and normal["pid"] == os.getpid()


现在把同样的工作交给 Ray。`@ray.remote` 描述远程函数；每次 `.remote()` 提交一个 Task。
返回的 `ObjectRef` 引用结果，结果可能还未产生。worker 执行计算，driver 在 `ray.get()` 处等待并取回值。
执行失败也会在取结果时传播。[Tasks](https://docs.ray.io/en/latest/ray-core/tasks.html)

**预测：** `ref` 是整数 9 吗？worker PID 与 kernel PID 会有什么关系？


In [ ]:
@ray.remote(num_cpus=1)
def square(x, delay=0.15):
    started = event("task_start")
    time.sleep(delay)  # 教学等待，不是 CPU 算力负载。
    return {"input": x, "value": x * x, "events": [started, event("task_end")]}

with observe("one_task") as step:
    ref = square.remote(3)
    print("Returned reference:", ref)
    result = ray.get(ref, timeout=TIMEOUT)
    assert result["value"] == 9
    assert result["events"][0]["pid"] != os.getpid()
    step["result"] = result
    display(result)


**观察与复述：** 区分 `ref` 的类型、结果值和执行 PID。Task 可以在同机其他进程运行，也可以跨节点；每个 Task 并不一定新建一个进程。

```mermaid
---
config:
  theme: base
  themeVariables:
    background: '#FAF8F4'
    primaryColor: '#E4F0EB'
    primaryTextColor: '#292524'
    primaryBorderColor: '#0F766E'
    lineColor: '#6B635A'
    fontFamily: 'system-ui, sans-serif'
---
flowchart LR
    driver[Driver]:::communication -->|submit Task| scheduler[Scheduling]:::communication
    scheduler -->|dispatch| worker[Worker process]:::compute
    worker -->|produce| result[Result value]:::memory
    result -->|ray.get| driver
    driver -.->|ObjectRef| result
    classDef compute fill:#EAF0F8,stroke:#315C8C,color:#292524
    classDef memory fill:#F8EFDA,stroke:#9A6700,color:#292524
    classDef communication fill:#E4F0EB,stroke:#0F766E,color:#292524
```

图：提交和取回的简化模型；实线表示控制或数据流，虚线是引用关系。调度内部被省略，不表示单一全局调度器。

### 为什么过早等待会改变并发

如果 driver 提交一次就等一次，下一次提交发生在上一个结果之后。
先提交多个工作，再等待结果，才会留下多个可调度任务；实际运行并发仍取决于资源与依赖。

**预测：** 8 个任务、4 个逻辑 CPU、每个任务声明 1 CPU，两种写法的执行区间怎样不同？等待时长 0.15 秒是教学参数。


In [ ]:
TASK_COUNT = 8
DELAY_SECONDS = 0.15
run["parameters"].update(task_count=TASK_COUNT, teaching_delay_seconds=DELAY_SECONDS)

with observe("submit_then_get_each") as step:
    started = time.perf_counter_ns()
    sequential_results = [ray.get(square.remote(i, DELAY_SECONDS), timeout=TIMEOUT) for i in range(TASK_COUNT)]
    step["driver_elapsed_ns"] = time.perf_counter_ns() - started
    step["results"] = sequential_results
    assert [r["value"] for r in sequential_results] == [i * i for i in range(TASK_COUNT)]
    display(step)


先从上面的事件看：下一任务能否在上一个结束前开始？再把等待移到所有提交之后。
`ray.get(list_of_refs)` 返回与引用列表对应的值；不要把返回列表的顺序当作完成顺序。


In [ ]:
with observe("submit_all_then_get") as step:
    started = time.perf_counter_ns()
    refs = [square.remote(i, DELAY_SECONDS) for i in range(TASK_COUNT)]
    submitted_results = ray.get(refs, timeout=TIMEOUT)
    step["driver_elapsed_ns"] = time.perf_counter_ns() - started
    step["results"] = submitted_results
    assert [r["value"] for r in submitted_results] == [i * i for i in range(TASK_COUNT)]
    display(step)


### 用实际事件画时间线

下面每根条是一次调用的开始到结束，包含本例的教学等待。横轴是本组首个任务开始后的秒数；标签保留 Task、PID 与线程信息。
两组各自归零，不能把它们解释成同一批同时提交的任务。

先看重叠关系，再看总耗时。冷启动、调度、序列化与系统负载都进入 driver 耗时，不能断言第二组一定更快。
[循环中的 get](https://docs.ray.io/en/latest/ray-core/patterns/ray-get-loop.html) · [过细粒度任务](https://docs.ray.io/en/latest/ray-core/patterns/too-fine-grained-tasks.html)


In [ ]:
import matplotlib.pyplot as plt

plt.style.use(REPO / "assets/styles/plots.mplstyle")
colors = json.loads((REPO / "assets/styles/tokens.json").read_text())["colors"]

def show_timeline(groups, name):
    fig, axes = plt.subplots(len(groups), 1, figsize=(9, 3 * len(groups)), squeeze=False)
    for ax, (title, results) in zip(axes[:, 0], groups):
        origin = min(r["events"][0]["at_ns"] for r in results)
        labels = []
        for i, result in enumerate(results):
            start, end = result["events"][0], result["events"][-1]
            ax.barh(i, (end["at_ns"] - start["at_ns"]) / 1e9,
                    left=(start["at_ns"] - origin) / 1e9, color=colors["compute"])
            labels.append(f"Call {result['input']} · PID {start['pid']} · Thread {start['thread_id']}")
        ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel("Time since first start (s)")
        ax.set_ylabel("Call / process / thread")
        ax.set_title(title)
    fig.suptitle("Ray calls · measured event intervals, including teaching waits")
    fig.savefig(RUN_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(RUN_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

with observe("task_timeline"):
    show_timeline([("Submit then get each", sequential_results),
                   ("Submit all then get", submitted_results)], "task-timeline")


**复述：** 结果一样，控制流哪里不同？哪些间隔来自等待，哪些成本这张图没有分开？

## 2 · ObjectRef：让依赖跟着引用走

任务的返回值与 `ray.put(value)` 都能产生对象引用。顶层 ObjectRef 参数建立依赖，执行函数时会解析成对应值。
所以可以把生产者的引用直接传给消费者，driver 无需先拿回中间结果。[Objects](https://docs.ray.io/en/latest/ray-core/objects.html)

**预测：** 下面消费者的 `value` 参数是引用还是字典？它能在平方结果产生前执行吗？


In [ ]:
@ray.remote(num_cpus=1)
def plus_one(value):
    return {"value": value["value"] + 1, "events": [event("dependency_consumed")]}

with observe("top_level_dependency") as step:
    producer_ref = square.remote(3)
    consumer_ref = plus_one.remote(producer_ref)
    consumer_result = ray.get(consumer_ref, timeout=TIMEOUT)
    producer_result = ray.get(producer_ref, timeout=TIMEOUT)
    assert consumer_result["value"] == 10
    step["producer"] = producer_result
    step["consumer"] = consumer_result
    display(step)


这里观察的是顶层参数；嵌套在列表或字典中的引用，不能直接套用同样的自动解析规则。
Ray 对象按不可变对象理解；大小、类型和节点位置会改变序列化与传输成本，不是所有对象都能零拷贝。

### 一个输入，多个消费者

`ray.put()` 让多个工作引用同一个输入，避免反复按值提交同一份大参数；跨节点数据仍可能需要传输。
下面用小列表核查语义，不测试大对象性能。[重复大参数](https://docs.ray.io/en/latest/ray-core/patterns/pass-large-arg-by-value.html)

**预测：** 消费者收到的参数是什么？它们各自归约后的值应相同吗？


In [ ]:
@ray.remote(num_cpus=1)
def total(values):
    return {"value": sum(values), "events": [event("object_consumed")]}

with observe("shared_input") as step:
    values = list(range(16))
    shared_ref = ray.put(values)
    consumer_results = ray.get([total.remote(shared_ref) for _ in range(2)], timeout=TIMEOUT)
    assert [r["value"] for r in consumer_results] == [sum(values)] * 2
    step["results"] = consumer_results
    display(consumer_results)


### 背压：限制在途工作，不等同于限制运行资源

生产者提交过快会积累工作。`ray.wait()` 返回已就绪与剩余的引用，用它收集部分结果后再提交，能控制未收集工作数。
调度可同时运行多少任务，主要由任务资源要求与可用资源决定。[在途任务限制](https://docs.ray.io/en/latest/ray-core/patterns/limit-pending-tasks.html)

**预测：** 4 个 CPU、在途上限 3 时，记录中的 pending 数量能达到 8 吗？`ready` 是数值结果吗？


In [ ]:
with observe("bounded_pending_work") as step:
    pending, completed, pending_counts = [], [], []
    PENDING_LIMIT = 3
    for i in range(TASK_COUNT):
        if len(pending) == PENDING_LIMIT:
            ready, pending = ray.wait(pending, num_returns=1, timeout=TIMEOUT)
            if not ready:
                raise TimeoutError("没有任务在等待上限内就绪")
            completed.extend(ray.get(ready, timeout=TIMEOUT))
        pending.append(square.remote(i, DELAY_SECONDS))
        pending_counts.append(len(pending))
    completed.extend(ray.get(pending, timeout=TIMEOUT))
    assert sorted(r["value"] for r in completed) == [i * i for i in range(TASK_COUNT)]
    step.update(pending_limit=PENDING_LIMIT, pending_counts=pending_counts, results=completed)
    display({"pending_limit": PENDING_LIMIT, "counts_after_submission": pending_counts})


**复述：** 为什么限制未收集引用可以形成背压？它与 `num_cpus` 表达的约束分别发生在哪一层？

## 3 · Actor：实例状态留在执行进程

重复调用要复用模型、缓存或计数器时，用持久实例比独立函数更自然。
`@ray.remote` 修饰类后，`.remote()` 创建 Actor；方法调用也用 `.remote()`，返回结果引用。[Actors](https://docs.ray.io/en/latest/ray-core/actors.html)

默认同步 Actor 串行执行方法。下面从单个 driver 提交三次调用，显式声明一个 CPU。
**预测：** 实例状态会在调用之间继续累积吗？三个返回值的 PID 应怎样？


In [ ]:
@ray.remote(num_cpus=1)
class Counter:
    """在一个持久 Actor 进程中保留计数状态。"""
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return {"value": self.value, "events": [event("counter_increment")]}

with observe("persistent_actor_state") as step:
    counter = Counter.remote()
    try:
        counter_results = ray.get([counter.increment.remote() for _ in range(3)], timeout=TIMEOUT)
        assert [r["value"] for r in counter_results] == [1, 2, 3]
        assert len({r["events"][0]["pid"] for r in counter_results}) == 1
        step["results"] = counter_results
        display(counter_results)
    finally:
        ray.kill(counter)


**观察与复述：** 对照 Task 的 PID，解释 Actor 保留了什么。
本例的顺序依赖默认同步 Actor、单提交者和成功执行，不能泛化成所有配置的全局顺序。
Actor 进程内状态也不是外部持久存储；重启、重试、checkpoint 与副作用语义需要分别考虑。
[执行顺序](https://docs.ray.io/en/latest/ray-core/actors/task-orders.html) · [故障恢复](https://docs.ray.io/en/latest/ray-core/fault_tolerance/actors.html)

## 4 · 串行、线程与协程：状态共享之后怎样推进工作

先看默认 Actor。方法在一个进程内排队执行；下面的 `time.sleep()` 模拟等待。
锁保护计数更新，后面同一个同步类还会用于 Threaded Actor。[Actor 并发](https://docs.ray.io/en/latest/ray-core/actors/async_api.html)

**预测：** 一次提交 4 个方法调用，是否就意味着 4 个方法同时执行？


In [ ]:
@ray.remote(num_cpus=1)
class WaitingActor:
    """保留计数，并为串行和线程配置安全更新共享状态。"""
    def __init__(self):
        self.completed = 0
        self.lock = threading.Lock()

    def wait(self, number):
        started = event("actor_call_start")
        time.sleep(DELAY_SECONDS)
        with self.lock:
            self.completed += 1
            completed = self.completed
        return {"input": number, "completed": completed, "events": [started, event("actor_call_end")]}

with observe("serial_actor") as step:
    actor = WaitingActor.remote()
    try:
        serial_results = ray.get([actor.wait.remote(i) for i in range(4)], timeout=TIMEOUT)
        assert sorted(r["completed"] for r in serial_results) == [1, 2, 3, 4]
        step["results"] = serial_results
        display(serial_results)
    finally:
        ray.kill(actor)


### Threaded Actor：同一进程中的线程池

同步方法配合 `max_concurrency=2` 使用线程并发。不同线程共享实例，计数器更新需要锁。
在传统、启用 GIL 的 CPython 中，线程并发不保证纯 Python 计算并行；这里的 sleep 能交叠等待，部分原生计算库也会释放 GIL。

**预测：** PID 应相同还是不同？线程 ID 与时间区间怎样变化？


In [ ]:
with observe("threaded_actor") as step:
    actor = WaitingActor.options(max_concurrency=2).remote()
    try:
        threaded_results = ray.get([actor.wait.remote(i) for i in range(4)], timeout=TIMEOUT)
        assert sorted(r["completed"] for r in threaded_results) == [1, 2, 3, 4]
        assert len({r["events"][0]["pid"] for r in threaded_results}) == 1
        step["results"] = threaded_results
        display(threaded_results)
    finally:
        ray.kill(actor)


### Async Actor：单线程事件循环，在 await 处让出执行

方法里的可等待操作使其他调用有机会推进。把阻塞计算包进 `async def` 不会自动并行，阻塞式 `ray.get()`/`ray.wait()` 也会阻塞事件循环。
需要等待 Ray 引用时使用异步等待。本例用 `asyncio.sleep()` 模拟等待，并刻意在“读计数 → 让出执行 → 写回”之间插入一次 `await`，用 `asyncio.Lock` 保护整个更新。若更新中没有 `await`，同一事件循环内的简单增量不需要按这个例子加锁。

**预测：** 为什么执行区间可以重叠，线程 ID 仍可能只有一个？


In [ ]:
@ray.remote(num_cpus=1)
class AsyncWaitingActor:
    """用事件循环交叠等待，并保护共享计数状态。"""
    def __init__(self):
        self.completed = 0
        self.lock = asyncio.Lock()

    async def wait(self, number):
        started = event("async_call_start")
        await asyncio.sleep(DELAY_SECONDS)
        async with self.lock:
            previous = self.completed
            await asyncio.sleep(0)  # 教学让出：锁保护跨 await 的读改写。
            self.completed = previous + 1
            completed = self.completed
        return {"input": number, "completed": completed, "events": [started, event("async_call_end")]}

with observe("async_actor") as step:
    actor = AsyncWaitingActor.options(max_concurrency=2).remote()
    try:
        async_results = ray.get([actor.wait.remote(i) for i in range(4)], timeout=TIMEOUT)
        assert sorted(r["completed"] for r in async_results) == [1, 2, 3, 4]
        assert len({r["events"][0]["thread_id"] for r in async_results}) == 1
        step["results"] = async_results
        display(async_results)
    finally:
        ray.kill(actor)


### 把三个执行模型放在同一组图中

不同进程各有地址空间；线程共享一个进程中的状态；协程共享事件循环和状态，在让出执行的位置交叠推进。
若把“读取旧计数 → await → 写回”分开，即便只有一个线程，也可能出现丢失更新。
下面的图只根据已运行事件生成，条形包含教学等待。[Async Actor pattern](https://docs.ray.io/en/latest/ray-core/patterns/concurrent-operations-async-actor.html)


In [ ]:
with observe("actor_timeline"):
    show_timeline([("Serial actor", serial_results), ("Threaded actor · max concurrency 2", threaded_results),
                   ("Async actor · max concurrency 2", async_results)], "actor-timeline")


**复述：** 分别用 PID、线程 ID、等待位置和共享状态解释三种结果。为什么这些数据不能说明线程提高了纯 Python 算力？

## 5 · 逻辑资源与后续连接

`num_cpus=1` 表达准入需求，不是绑定一个物理核或限制库只用一个线程。
本页 Actor 显式声明 CPU，避免默认 Actor 的历史资源规则干扰观察。
下面核查本运行时资源，再进入 GPU Notebook。[资源模型](https://docs.ray.io/en/latest/ray-core/scheduling/resources.html)


In [ ]:
with observe("logical_resources") as step:
    step["resources"] = ray.cluster_resources()
    display({"ray_logical_resources": step["resources"],
             "host_cpu_count": run["environment"]["host_cpu_count"],
             "kernel_pid": os.getpid()})


Ray 的 Task/Actor 负责执行编排；[NCCL](../../distributed-training/README.md)关注张量通信，
[vLLM](../../inference-serving/README.md)关注生成循环、请求和 KV 调度，
[verl](../../../paths/ai-infra/14-post-training-rlhf.md#ai-14-7)把后训练角色与这些执行机制组合。
这些是职责定位，尚未追踪特定版本代码。Ray Actor 是有状态进程机制，RL 中 actor 常指 policy 角色。


## 保存、清理与复述

先回答本页的观察问题，写下自己的解释和遗漏前提，再执行清理单元。步骤的 `passed` 表示该运行单元完成；会话的 `finished` 表示已保存并清理，不代表所有单元已执行或已经掌握。
重新开始一个完整实验时，重启 kernel 并从头运行，使用新的运行目录。


In [ ]:
if owns_runtime:
    ray.shutdown()
    owns_runtime = False
run["status"] = "finished" if all(step["status"] == "passed" for step in run["steps"]) else "failed"
run["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
save_record()
display({"status": run["status"], "record": str(RUN_DIR / "run.json")})


### 回收学习成果

保存 Notebook 的真实输出；把本次 `.runs/` 目录和 Notebook 一起取回本机。服务器修改了代码或讲解时，先比较差异，再合并。
例如 SSH 别名为 `gpu-box`、服务器仓库为 `~/ai-infra-learning`，在本机仓库根目录执行：

```bash
mkdir -p .runs/from-ray-server
rsync -av gpu-box:~/ai-infra-learning/topics/distributed-runtime/ray/cpu.ipynb .runs/from-ray-server/
rsync -av gpu-box:~/ai-infra-learning/.runs/ray-cpu/ .runs/from-ray-server/ray-cpu/
```

精选记录放主题 `results/`；图保留 SVG 与绘图单元，Notebook 保存 PNG 输出。你的复述填在下面，未解决的问题进入 [GAPS](../../../GAPS.md)。

- **我的原先解释：** 待填写。
- **实际观察与记录位置：** 未运行，待填写。
- **修正后的解释与适用前提：** 待填写。
- **下一步问题：** 待填写。


## 来源与边界

Ray 官方资料按 2.59.0 核查，日期 2026-10-02；`latest` 后续可能变化，以运行记录中的安装版本为准。
上面的链接分别支持 Task、对象、Actor、并发与资源结论，代码和预测问题为本仓库组织。

[用户提供的视频](https://www.bilibili.com/video/BV1HuZcBEEyy/)标题与简介已核查；正文、字幕和私有示例代码未读取。
本 Notebook 依据官方资料，不作为视频逐段总结。实验未运行，不预设性能排序或个人掌握度。
